# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 2, Refresh / Content Opportunity Scoring.** The question is the same as in W01: which pages should a reviewer look at first? This week I audit my own W05 model the way the live session audited the FlyRank research paper. Same frame as W03 to W05: **one row = one page**, features from March 2026 and the label `declined_next_month` (April impressions per day more than 20% below March). My W05 model is a logistic regression on the five W03 features, trained and scored inside the review pool (pages with 5+ impressions a day in late March).

**Short version of what I found:**
- **Grouping by client was not the weak spot of W05.** My logistic regression gives almost the same P@50 with a plain random row split (0.880 vs 0.884), so it was not memorizing clients. Gradient boosting was, a little (ROC AUC 0.733 with random rows vs 0.709 grouped).
- **Time was the weak spot.** W05 trained on April outcomes of other clients, which nobody has on 31 March. Trained only on what was known by 31 March, my model's P@50 on the March→April test falls from 0.884 to 0.776. That is level with the W04 rule (0.784) and below a plain sort by the late-March slide (0.800). One month later (April→May) the time-aware model holds up (0.988 vs 0.884 for the rule), but it still only ties the slide sort (0.980).
- **The leakage audit found no feature that reads the future.** It did find that the label shares March with the features: 43% of the W05 top-50 hits had already dropped in late March and did not fall another 20% in April. On a stricter label (did the page keep falling after 31 March?) the model is at the base rate.

**How to run it:** same as W03 to W05. In Colab save your Hugging Face READ token as a Secret named `HF_TOKEN`. Outside Colab the notebook reads the `HF_TOKEN` environment variable or asks for it with `getpass`. The token is never printed and never written into the notebook. Every random step uses seed 42, so a rerun gives the same tables. The notebook reads the February to May partitions of the daily table. June, the sealed month, is never read.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import getpass
import platform
from datetime import date
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, ndcg_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# The repo is public, so the token comes from a Colab Secret or an environment variable.
# It is never printed. getpass is only the last fallback.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET enable_progress_bar = false")     # keeps the saved outputs clean
print("DuckDB connected. Hugging Face token loaded (not printed).")
VERSIONS = {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "pandas": pd.__version__,
            "numpy": np.__version__, "duckdb": duckdb.__version__}
print("Versions:", " | ".join(f"{k} {v}" for k, v in VERSIONS.items()))

DuckDB connected. Hugging Face token loaded (not printed).
Versions: python 3.13.17 | scikit-learn 1.9.1 | pandas 3.0.5 | numpy 2.5.3 | duckdb 2.0.0


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The paper is `docs/flyrank-seo-research-march-2026.pdf` (*The State of AI-Driven SEO in Numbers*, March 2026, 36 pages). It was written for a broad audience and it is open about its own limits. It keeps reversed and nuanced results visible, drops the unstable `361+` freshness bucket (283:1 from a single declining page), points out survivor bias in the small `365+ × 361+` cell, warns that its health score is partly built from the same inputs its Random Forest uses and marks every ML page as exploratory. So my job is not to grade it.

I picked the two findings closest to my lane (which pages to refresh, which pages will decline). For each one I ask the two questions from the live session (**where does the label or outcome come from?** and **does the validation design carry the claim?**) and then write what would make the claim stronger. Page numbers are the numbers printed in the paper's footer.

### Finding A: the refresh effect (Finding #4 on p.9 and playbook priority #1 on p.31)

**What the paper says.** Pages older than 365 days that were updated in the last 30 days show a "3.2x health boost (from 10.7 to 34.5) and 57x more impressions (from 71 to 4039)". The playbook turns this into its first action ("Refresh mature pages before they decay") and the action card expects a refresh to lift "mature pages from roughly 10.7 health to 34.5".

**Question 1: where does the outcome come from?**
- Health is not a separate outcome from impressions. By the paper's own definition (p.5) 30 of the 100 health points come from impressions and 30 from position. So "3.2x health" and "57x impressions" are partly the same gap counted twice.
- The paper says its performance numbers use a rolling 90-day window (p.4). If the impressions in this comparison use that window, then for a page updated in the last 30 days at least 60 of the 90 days come **before** the update. Most of the 4,039 average impressions would then have been earned before the refresh happened. My question: were these pages already much bigger than the untouched ones before they were refreshed?

**Question 2: does the design carry the claim?**
- It is one snapshot that compares two different groups of pages: old pages someone chose to refresh and old pages nobody touched. It is not the same pages before and after. Teams usually refresh the pages that still matter to them, so part of the gap can come from which pages were picked and not from what the refresh did. The words "boost" and "lifts" read like a before/after effect on the same page, which this design can't show.
- 71 vs 4,039 are averages of a very heavy-tailed number. In my W04 audit the top 1% of pages held 18% of all impressions, so a few giant pages can produce a ratio like 57x. A median or the share of pages that grew would be steadier.

**What would make it stronger.** The paper's own "Measure" line already names the right test: refreshed pages vs comparable untouched pages. Concretely: for each refreshed page compare its impressions in the weeks before and after the edit date. Then do the same for untouched pages of the same client and age over the same calendar weeks (a difference-in-differences). One practical catch I hit in W03: a snapshot only keeps the *last* edit date (our `dim_content` is a July 2026 snapshot), so this test needs the edit history, not only the latest date.

**Why it matters for my lane.** My queue also says "look at these pages first" and my label can't show that a refresh would help either. Same gap, so I need the same careful wording (section 4).

### Finding B: the growth model (ML appendix "What Predicts Growth?", p.29)

**What the paper says.** A logistic regression with "71% holdout accuracy" separates growing from declining pages. Content age is the strongest negative signal, while "days visible and recent impressions are among the strongest positive signals". The methodology page (p.36) says it used an 80/20 split on 61.8K active pages.

**Question 1: where does the label come from and do the features share its window?**
- Growing vs declining comes from the trend direction: impressions in the last 30 days vs the 30 days before (up = more than 10% growth, down = more than 10% decline, p.5).
- The features include impressions, clicks, sessions and days visible. Elsewhere in the paper these are 90-day numbers ("days visible in 90", p.10). If the model used those, then 60 of the 90 feature days sit inside the two 30-day windows that the label compares. A page that grew in the last 30 days gets more impressions and more visible days in the 90-day window just because it grew. That is the "overlapping windows" leak from our skill library. It would make "recent impressions" and "days visible" look strong for a mechanical reason. My question: did every feature window end before the label's first 30-day window starts?

**Question 2: does the validation design carry the claim?**
- The 80/20 split is not described further. The portfolio has 57 brands and pages of one brand share a lot (in my W03 data the April decline rate ran from 27% to 94% across the big clients). With a random row split, pages of the same brand sit on both sides. A split grouped by brand, or a time split, would answer "does it work on a brand or a month it never saw?".
- 71% accuracy needs its base rate next to it. The paper doesn't print the class mix of the ML sample. In Finding #1 the rising and falling pages are 74,187 vs 45,272, so always saying "growing" would already score 62.1%. If the ML sample has a similar mix, the model adds about 9 points, not 71.

**What would make it stronger.** Print the base rate next to the accuracy, build the features only from days before the label window and report a brand-grouped or time-based score next to the random one.

**Both questions can be tested on my own model, so I do.** Section 2 compares a random row split with grouped and time-aware splits. Section 3 puts window-overlap features like "90-day impressions" and "days visible" into my model on purpose to see how much they inflate the score. The cell below only does the arithmetic on the paper's own numbers.

In [3]:
# Numbers copied by hand from docs/flyrank-seo-research-march-2026.pdf (p. = page number in the paper's footer)
rising, falling = 74_187, 45_272                    # Finding #1, p.6: pages with rising vs falling impressions
growth_model_accuracy = 0.71                       # ML appendix, p.29: logistic regression, 80/20 holdout
stale_imp, refreshed_imp = 71, 4_039               # Finding #4, p.9: 365+ day pages, untouched vs refreshed in the last 30 days
stale_health, refreshed_health = 10.7, 34.5        # same two groups, health score
HEALTH_POINTS = {"impressions": 30, "position": 30, "CTR": 20, "scroll depth": 20}   # p.5
WINDOW_DAYS, TREND_DAYS, FRESH_DAYS = 90, 30, 30   # p.4-5: 90-day performance window, 30d vs previous 30d trend

always_growing = rising / (rising + falling)
paper_math = pd.DataFrame([
    ("Finding #4", "impressions, refreshed vs untouched 365+ pages", f"{refreshed_imp / stale_imp:.1f}x",
     "the paper's 57x"),
    ("Finding #4", "health, refreshed vs untouched 365+ pages", f"{refreshed_health / stale_health:.2f}x",
     "the paper's 3.2x"),
    ("Finding #4", "health points that come from impressions and position", f"{HEALTH_POINTS['impressions'] + HEALTH_POINTS['position']} of 100",
     "so health and impressions are not two separate results"),
    ("Finding #4", "days of a 90-day window that come BEFORE a refresh made in the last 30 days", f"at least {WINDOW_DAYS - FRESH_DAYS} of {WINDOW_DAYS}",
     "if the impressions are the paper's 90-day window"),
    ("ML appendix", "accuracy of always saying 'growing' (rising share in Finding #1)", f"{always_growing:.1%}",
     "the base rate the 71% should sit next to"),
    ("ML appendix", "skill above that base rate", f"{100 * (growth_model_accuracy - always_growing):.1f} points",
     "if the ML sample has a similar mix"),
    ("ML appendix", "days of the 90-day feature window inside the two 30-day trend windows", f"{2 * TREND_DAYS} of {WINDOW_DAYS}",
     "if impressions / days visible use the 90-day window"),
], columns=["finding", "what I computed", "value", "note"])
paper_math

,finding,what I computed,value,note
0,Finding #4,"impressions, refreshed vs untouched 365+ pages",56.9x,the paper's 57x
1,Finding #4,"health, refreshed vs untouched 365+ pages",3.22x,the paper's 3.2x
2,Finding #4,health points that come from impressions and p...,60 of 100,so health and impressions are not two separate...
3,Finding #4,days of a 90-day window that come BEFORE a ref...,at least 60 of 90,if the impressions are the paper's 90-day window
4,ML appendix,accuracy of always saying 'growing' (rising sh...,62.1%,the base rate the 71% should sit next to
5,ML appendix,skill above that base rate,8.9 points,if the ML sample has a similar mix
6,ML appendix,days of the 90-day feature window inside the t...,60 of 90,if impressions / days visible use the 90-day w...


**The two findings side by side:**

| | Finding A: refresh effect | Finding B: growth model |
|---|---|---|
| Where the outcome / label comes from | health (60 of 100 points are impressions and position) and impressions, probably over a 90-day window that is mostly before the refresh | trend direction: last 30 days vs the 30 days before |
| Does the design carry the claim? | one snapshot, two different groups of pages, picked for refresh by people: it shows an association, not a lift | 80/20 split with no mention of grouping by brand or time, no base rate (about 62% if the mix matches Finding #1), features may share the label window |
| What would make it stronger | before/after on the same pages vs matched untouched pages, medians next to averages | features cut off before the label window, brand-grouped or time split, base rate printed |
| Safe wording today | "older pages that were recently updated showed much higher impressions in this snapshot" | "in this sample, a model separating growing and declining pages leaned on age and visibility (exploratory)" |

The paper already uses safe wording in many places ("the safe reading is narrower", "descriptive rather than causal"). For Finding A my question is mostly about the action card, where the snapshot gap turns into an expected lift. For Finding B the paper already calls the model descriptive, so my question is about the 71% number itself.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before this week:** W05 already used a split that looks honest, `GroupKFold` by client, so every test page came from a client the model never saw. But grouping does not close a second path: **time**. In W05 the model trained on the March→April pages of the other clients, so it learned from **April outcomes**. On 31 March, the day the queue would be built, nobody has any April outcome. If April had something in common across clients (a seasonal dip or a search update, for example), the W05 model could pick it up from other clients and look better than it would in real use.

**This week:** I keep the model exactly as in W05 (logistic regression, the five W03 features, the review pool) and only change what it may learn from. Every design is scored on **the same test pages**: the 84,592 pool pages of March→April, in the same five client folds as W04 and W05. So any change in the numbers comes from the training data and nothing else.

| Design | Trains on | Sees April outcomes? | Question it answers |
|---|---|---|---|
| A. random rows (reference) | 80% of the Mar→Apr pages, picked at random | yes, even of the same clients | what a plain "80/20 split" would report |
| B. grouped by client (W05, **before**) | Mar→Apr pages of the other clients | yes, of other clients | does it work on a new client if April were already known? |
| C. time-aware (**after**) | Feb→Mar pages of all clients (labels complete on 31 March) | no | does it work next month for the clients we already have? |
| D. time-aware + grouped (**after**, strictest) | Feb→Mar pages of the other clients only | no | does it work next month on a client it never saw? |

The Feb→Mar slice is the check slice from the W04 signal audit, rebuilt with the same rules. Gradient boosting runs next to the logistic regression again, because W05 said its steadier folds were worth a second look here. Then a **second time step** repeats the test one month later (train on Mar→Apr, test on Apr→May), so the result does not rest on a single month pair.

### Rebuild the frames and check them against W04 / W05

One query per month (February to May) sums each page's daily rows with the same sums as W03 to W05. Then one function builds a page frame for any pair of months with the four W03 eligibility rules. For March→April it must give back the exact W05 frame and review pool.

In [4]:
REL = "hf://datasets/FlyRank/internship-warehouse"

# Same contract as W03, W04 and W05
LABEL = "declined_next_month"         # label month per day more than 20% below the feature month average (W03)
STRICT_LABEL = "fell_after_decision"  # label month per day more than 20% below the LATE half of the feature month (section 3)
DECLINE_CUTOFF_PCT = -20
MIN_IMPRESSIONS = 100                 # W03 eligibility
MIN_SLIDE_PCT, MIN_LATE_PER_DAY = 20, 5   # the W04 rule, unchanged. The 5-a-day bar is also the W05 review pool
SEED, N_FOLDS, K_MAIN = 42, 5, 50

# Second half of each month starts on this day (same halves as W03 to W05: 1-15 / 16-31 March, 1-14 / 15-28 February)
LATE_HALF_START = {"2026-02": "2026-02-15", "2026-03": "2026-03-16", "2026-04": "2026-04-16", "2026-05": "2026-05-16"}
DECISION_MOMENT = pd.Timestamp("2026-03-31")   # end of this day, for the main test

# The W05 model, unchanged. W05 called the 4th feature late_march_share. Here it is the share of the
# feature month's impressions that came in its second half, so the same code works for every month.
FEATURES = ["imp_per_day_log", "ctr_pct", "avg_position", "late_half_share", "gsc_days"]
MAIN, CHECK = "logistic regression", "gradient boosting"


def logistic_regression():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


def gradient_boosting():
    return HistGradientBoostingClassifier(random_state=SEED)


MODELS = {MAIN: logistic_regression, CHECK: gradient_boosting}
RULE, SLIDE_ONLY, RANDOM = "W04 rule (baseline)", "slide only (no model)", "random order"
print("Model:", MAIN, "on", ", ".join(FEATURES), "| check model:", CHECK)

Model: logistic regression on imp_per_day_log, ctr_pct, avg_position, late_half_share, gsc_days | check model: gradient boosting


In [5]:
def month_pages(month):
    """One row per page for one month partition. Same sums as the W03 / W05 query, plus two columns
    (days with impressions, first 7 days) that only the section 3 leak trap reads."""
    return con.sql(f'''
        SELECT
            content_hash_id,
            ANY_VALUE(client_hash_id)                           AS client_hash_id,  -- safe: W03 query 1 showed one client per page
            COUNT(*)                                            AS n_rows,
            SUM(gsc_impressions)::BIGINT                        AS imp,
            SUM(gsc_clicks)::BIGINT                             AS clicks,
            SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '{LATE_HALF_START[month]}')::BIGINT AS imp_late,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_days,
            SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)::BIGINT AS position_sum,
            SUM(gsc_impressions)  FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)::BIGINT AS position_impressions,
            COUNT(*) FILTER (WHERE gsc_impressions > 0)         AS days_with_impressions,
            SUM(gsc_impressions) FILTER (WHERE report_date < DATE '{month}-08')::BIGINT AS imp_first_7_days,
            MIN(report_date)                                    AS first_date,
            MAX(report_date)                                    AS last_date
        FROM read_parquet('{REL}/fact_content_daily_performance/month={month}/*.parquet')
        GROUP BY content_hash_id
    ''').df()


MONTHS = {m: month_pages(m) for m in LATE_HALF_START}      # February to May. June (sealed) is never read
clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM read_parquet('{REL}/dim_clients.parquet')").df()
pd.DataFrame({m: {"pages": len(d), "first day read": d["first_date"].min(), "last day read": d["last_date"].max()}
              for m, d in MONTHS.items()}).T

,pages,first day read,last day read
2026-02,321546,2026-02-01 00:00:00,2026-02-28 00:00:00
2026-03,331437,2026-03-01 00:00:00,2026-03-31 00:00:00
2026-04,362172,2026-04-01 00:00:00,2026-04-30 00:00:00
2026-05,389153,2026-05-01 00:00:00,2026-05-31 00:00:00


In [6]:
def page_frame(feature_month, label_month):
    """One row per page with the W03 rules: features from feature_month only, labels from label_month only.
    Returns the frame and a hindsight table (label-month numbers) that is only read after scoring."""
    f, lab = MONTHS[feature_month], MONTHS[label_month]
    first_day = pd.Timestamp(f"{feature_month}-01")
    f_days, l_days = first_day.days_in_month, pd.Timestamp(f"{label_month}-01").days_in_month
    early_days = (pd.Timestamp(LATE_HALF_START[feature_month]) - first_day).days
    late_days = f_days - early_days
    raw = (f.merge(clients, on="client_hash_id", how="left")
            .merge(lab[["content_hash_id", "n_rows", "imp"]].rename(columns={"n_rows": "label_rows", "imp": "imp_label"}),
                   on="content_hash_id", how="left"))
    counts = ["n_rows", "imp", "clicks", "imp_late", "gsc_days", "position_sum", "position_impressions", "label_rows", "imp_label"]
    raw[counts] = raw[counts].astype("float64")           # SQL NULL becomes NaN
    rules = [raw["n_rows"].eq(f_days),                                        # 1. a row on every day of the feature month
             pd.to_datetime(raw["gsc_data_start"]).le(first_day),             # 2. client GSC tracking started by day 1
             raw["imp"].ge(MIN_IMPRESSIONS),                                  # 3. at least 100 impressions
             raw["label_rows"].eq(l_days)]                                    # 4. a row on every day of the label month
    keep = np.logical_and.reduce(rules)
    removed_only_by_rule_4 = int((np.logical_and.reduce(rules[:3]) & ~keep).sum())   # section 3: does rule 4 select on the future?
    e = raw[keep].sort_values("content_hash_id").reset_index(drop=True)

    d = pd.DataFrame({
        "content_hash_id": e["content_hash_id"],
        "client_hash_id": e["client_hash_id"],
        # the five W03 / W05 features (feature month only)
        "imp_per_day_log": np.log1p(e["imp"] / f_days),
        "ctr_pct": 100 * e["clicks"] / e["imp"],
        "avg_position": e["position_sum"] / e["position_impressions"],
        "late_half_share": e["imp_late"] / e["imp"],
        "gsc_days": e["gsc_days"] * 31 / f_days,          # scaled to a 31-day month, so March is unchanged
        # W04 rule inputs and review columns (feature month only)
        "imp_per_day": e["imp"] / f_days,
        "early_per_day": (e["imp"] - e["imp_late"]) / early_days,
        "late_per_day": e["imp_late"] / late_days,
        "clicks_per_day": e["clicks"] / f_days,
    })
    d["slide_pct"] = 100 * (1 - d["late_per_day"] / d["early_per_day"].where(d["early_per_day"] > 0))
    d["in_pool"] = d["late_per_day"] >= MIN_LATE_PER_DAY            # the W05 review pool

    # Labels: the label month vs the feature month. The % changes stay OUT of the frame
    label_per_day = e["imp_label"] / l_days
    change_pct = 100 * (label_per_day - d["imp_per_day"]) / d["imp_per_day"]
    vs_late_pct = 100 * (label_per_day - d["late_per_day"]) / d["late_per_day"].where(d["late_per_day"] > 0)
    d[LABEL] = (change_pct < DECLINE_CUTOFF_PCT).astype(int)
    d[STRICT_LABEL] = (vs_late_pct < DECLINE_CUTOFF_PCT).astype(int)
    hindsight = pd.DataFrame({"label_per_day": label_per_day.to_numpy(), "label_vs_month_pct": change_pct.to_numpy(),
                              "label_vs_late_half_pct": vs_late_pct.to_numpy()}, index=e["content_hash_id"])
    assert d["content_hash_id"].is_unique, "grain broken: expected one row per page"
    return d, hindsight, removed_only_by_rule_4


SLICES = {"Feb→Mar": ("2026-02", "2026-03"),   # time-aware training slice (its labels are complete on 31 March)
          "Mar→Apr": ("2026-03", "2026-04"),   # the W03 / W05 frame: main test
          "Apr→May": ("2026-04", "2026-05")}   # second time step
FRAMES, HINDSIGHT, RULE_4_REMOVED = {}, {}, {}
for name, (fm, lm) in SLICES.items():
    FRAMES[name], HINDSIGHT[name], RULE_4_REMOVED[name] = page_frame(fm, lm)
assert not set(FEATURES) & {LABEL, STRICT_LABEL}

pd.DataFrame({name: {"feature month": SLICES[name][0], "label month": SLICES[name][1], "pages": len(d),
                     "clients": d["client_hash_id"].nunique(), "base rate": round(d[LABEL].mean(), 4),
                     "pages in the review pool": int(d["in_pool"].sum()),
                     "base rate in the pool": round(d.loc[d["in_pool"], LABEL].mean(), 4)}
              for name, d in FRAMES.items()}).T

,feature month,label month,pages,clients,base rate,pages in the review pool,base rate in the pool
Feb→Mar,2026-02,2026-03,64881,26,0.2484,60290,0.2402
Mar→Apr,2026-03,2026-04,94496,38,0.5159,84592,0.4973
Apr→May,2026-04,2026-05,99868,44,0.5835,84691,0.5729


Three slices. Feb→Mar is smaller (64,881 pages, 26 clients) because fewer clients were tracked from 1 February. Its base rate is much lower (0.248 vs 0.516) because March was a stronger month than February for most sites (W04 audit). Apr→May has 99,868 pages from 44 clients and the highest base rate (about 0.58). In every slice the review pool keeps pages with 5+ impressions a day in the second half of the feature month, the same bar as W05.

One small change was needed to use other months: `gsc_days` is scaled to a 31-day month (×31/28 for February). For March the factor is 1, so the W05 features are unchanged. Also, a flat page has a second-half share of 0.500 in February and 0.516 in March. A fixed offset like that moves every score in a test month by the same amount, so it can't change the ranking of the logistic regression.

In [7]:
def sliding_with_demand(pages):
    # The W04 baseline rule, copied unchanged. Reads only two feature-month columns
    early, late = pages["early_per_day"], pages["late_per_day"]
    slide_pct = 100 * (1 - late / early.where(early > 0))
    fires = slide_pct.ge(MIN_SLIDE_PCT) & late.ge(MIN_LATE_PER_DAY)
    return pd.Series(np.where(fires, early - late, 0.0), index=pages.index)


def rank_order(scores, ids):
    # Highest score first. Ties are broken by content_hash_id, the same fixed order as the W04 queue
    return np.lexsort((np.asarray(ids), -np.asarray(scores, dtype=float)))


def queue_metrics(labels, scores, ids):
    y, s = np.asarray(labels), np.asarray(scores, dtype=float)
    top = y[rank_order(s, ids)]
    return {"P@20": top[:20].mean(), "P@50": top[:50].mean(), "P@100": top[:100].mean(),
            "avg precision": average_precision_score(y, s), "NDCG@50": ndcg_score(y[None, :], s[None, :], k=50),
            "ROC AUC": roc_auc_score(y, s)}


def client_folds(frame):
    # GroupKFold on ALL pages of a slice (like W04 / W05), grouped by client
    fold = np.full(len(frame), -1)
    for k, (_, test_idx) in enumerate(GroupKFold(n_splits=N_FOLDS).split(frame, frame[LABEL], groups=frame["client_hash_id"])):
        fold[test_idx] = k
    return fold


for name, d in FRAMES.items():
    d["rule_score"] = sliding_with_demand(d)
    d["fold"] = client_folds(d)
main, prev, nxt = FRAMES["Mar→Apr"], FRAMES["Feb→Mar"], FRAMES["Apr→May"]
pool = main[main["in_pool"]].reset_index(drop=True)            # the W05 review pool = the test pages of section 2
prev_pool = prev[prev["in_pool"]].reset_index(drop=True)
next_pool = nxt[nxt["in_pool"]].reset_index(drop=True)

# Receipts from W04 / W05 (fall back to the printed numbers if the files are not there, e.g. in Colab)
OUT_DIR = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs")
def receipt(name, fallback):
    p = OUT_DIR / name
    return json.loads(p.read_text()) if p.exists() else fallback
w05 = receipt("w05_model_metrics.json", {
    "data": {"pages": 94496, "clients": 38, "pool_pages": 84592, "pool_base_rate": 0.4973},
    "p50_per_fold": {"W04 rule (baseline)": [0.8, 0.92, 0.88, 0.7, 0.62], "logistic regression": [0.98, 0.96, 0.92, 0.82, 0.74],
                     "gradient boosting": [0.88, 0.94, 0.84, 0.92, 0.86]}})
w04_audit = receipt("w04_signal_audit_verdicts.json", {"slices": {"Feb→Mar (check)": {"pages": 64881, "clients": 26, "base_rate": 0.2484}}})

rule_p50 = [round(queue_metrics(g[LABEL], g["rule_score"], g["content_hash_id"])["P@50"], 3) for _, g in main.groupby("fold")]
same = {
    "Mar→Apr: same pages and clients as W05": (len(main), main["client_hash_id"].nunique()) == (w05["data"]["pages"], w05["data"]["clients"]),
    "Mar→Apr: same review pool and pool base rate as W05": (len(pool), round(pool[LABEL].mean(), 4)) == (w05["data"]["pool_pages"], w05["data"]["pool_base_rate"]),
    "Mar→Apr: same rule P@50 on each of the 5 client folds": rule_p50 == w05["p50_per_fold"][RULE],
    "Feb→Mar: same pages, clients and base rate as the W04 signal audit": (len(prev), prev["client_hash_id"].nunique(), round(prev[LABEL].mean(), 4))
        == tuple(w04_audit["slices"]["Feb→Mar (check)"][k] for k in ["pages", "clients", "base_rate"]),
}
for check, ok in same.items():
    print(f"{'PASS' if ok else 'FAIL'}  {check}")
assert all(same.values()), "a frame drifted from W04 / W05"

PASS  Mar→Apr: same pages and clients as W05
PASS  Mar→Apr: same review pool and pool base rate as W05
PASS  Mar→Apr: same rule P@50 on each of the 5 client folds
PASS  Feb→Mar: same pages, clients and base rate as the W04 signal audit


### What each design learns from

The cell below makes the design table concrete: how many training pages each design gets and the base rate of their labels. B trains on April labels (base rate 0.498), C and D on March labels (0.240). So C and D also learn from a month where far fewer pages declined. For P@50 that should matter less, because a queue only needs the order to be right.

In [8]:
DESIGNS = {
    "A": "random rows (reference)",
    "B": "grouped by client (W05, before)",
    "C": "time-aware (after)",
    "D": "time-aware + grouped (after, strictest)",
}


def training_pages(design, fold, test_pool, train_now, train_past):
    """What the model may learn from when it scores one held-out client fold of test_pool."""
    test_clients = set(test_pool.loc[test_pool["fold"] == fold, "client_hash_id"])
    if design == "B":
        return train_now[train_now["fold"] != fold]                               # same month, other clients
    if design == "C":
        return train_past                                                         # the month before, all clients
    if design == "D":
        return train_past[~train_past["client_hash_id"].isin(test_clients)]       # the month before, other clients only
    raise ValueError(design)


def out_of_fold(design, make_model, test_pool, train_past):
    scores = np.full(len(test_pool), np.nan)
    if design == "A":   # random rows: 5 shuffled folds over pages, so a client's pages sit on both sides
        for train_idx, test_idx in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(test_pool):
            model = make_model().fit(test_pool.loc[train_idx, FEATURES], test_pool.loc[train_idx, LABEL])
            scores[test_idx] = model.predict_proba(test_pool.loc[test_idx, FEATURES])[:, 1]
        return scores
    for fold in range(N_FOLDS):
        train = training_pages(design, fold, test_pool, test_pool, train_past)
        test = test_pool["fold"] == fold
        if design == "B":
            assert not set(train["client_hash_id"]) & set(test_pool.loc[test, "client_hash_id"]), "client on both sides"
        model = make_model().fit(train[FEATURES], train[LABEL])
        scores[test.to_numpy()] = model.predict_proba(test_pool.loc[test, FEATURES])[:, 1]
    return scores


def evaluate(test_pool, scores, label=LABEL):
    """Metrics per held-out client fold. Every method is measured on exactly the same test pages."""
    rows = []
    for fold in range(N_FOLDS):
        part = test_pool[test_pool["fold"] == fold]
        y, ids = part[label].to_numpy(), part["content_hash_id"].to_numpy()
        base = y.mean()
        rows.append({"fold": fold, "method": RANDOM, "P@20": base, "P@50": base, "P@100": base, "avg precision": base,
                     "NDCG@50": base, "ROC AUC": 0.5})       # expected values for a random queue
        for method, s in scores.items():
            rows.append({"fold": fold, "method": method, **queue_metrics(y, s[part.index], ids)})
    return pd.DataFrame(rows)


# What each design trains on (mean over the 5 test folds)
train_info = []
for code in ["B", "C", "D"]:
    sizes = [training_pages(code, k, pool, pool, prev_pool) for k in range(N_FOLDS)]
    train_info.append({"design": f"{code}. {DESIGNS[code]}",
                       "training labels from": "April (other clients)" if code == "B" else "March (known by 31 March)",
                       "sees any April outcome?": "yes" if code == "B" else "no",
                       "training pages (mean of folds)": int(np.mean([len(t) for t in sizes])),
                       "base rate of the training labels": round(float(np.mean([t[LABEL].mean() for t in sizes])), 3)})
train_info.insert(0, {"design": f"A. {DESIGNS['A']}", "training labels from": "April (same clients)", "sees any April outcome?": "yes",
                      "training pages (mean of folds)": int(len(pool) * (N_FOLDS - 1) / N_FOLDS),
                      "base rate of the training labels": round(pool[LABEL].mean(), 3)})
pd.DataFrame(train_info).set_index("design")

,training labels from,sees any April outcome?,training pages (mean of folds),base rate of the training labels
design,,,,
A. random rows (reference),April (same clients),yes,67673,0.497
"B. grouped by client (W05, before)",April (other clients),yes,67673,0.498
C. time-aware (after),March (known by 31 March),no,60290,0.240
"D. time-aware + grouped (after, strictest)",March (known by 31 March),no,48232,0.240


### My model under the four designs (March→April test)

For each design and each held-out client fold: train, score the fold's pool pages, rank them and measure. Same metrics as W05: **P@50** is the main number, with P@20, P@100, average precision, NDCG@50 and ROC AUC as support. Ties are broken by `content_hash_id` like in the W04 queue. The rule and the slide-only sort have nothing to train, so they are the same in every design. "Random order" is the fold's base rate. The first output line checks that design B gives back the W05 numbers exactly.

In [9]:
scores = {RULE: pool["rule_score"].to_numpy(), SLIDE_ONLY: -pool["late_half_share"].to_numpy()}   # nothing to train
for code in DESIGNS:
    for model_name, make_model in MODELS.items():
        scores[f"{model_name}, {code}"] = out_of_fold(code, make_model, pool, prev_pool)
by_fold = evaluate(pool, scores)

# W05 reproduced: design B must give the W05 fold numbers exactly
for model_name in MODELS:
    got = by_fold[by_fold["method"] == f"{model_name}, B"]["P@50"].round(3).tolist()
    assert got == w05["p50_per_fold"][model_name], f"{model_name} under design B does not match W05"
print("PASS  design B reproduces the W05 P@50 on every fold for both models")

ORDER = [RANDOM, RULE, SLIDE_ONLY] + [f"{m}, {c}" for m in MODELS for c in DESIGNS]
METRICS = ["P@20", "P@50", "P@100", "avg precision", "NDCG@50", "ROC AUC"]


def p50_table(by_fold, order):
    t = by_fold.pivot(index="method", columns="fold", values="P@50").loc[order]
    rule_row = t.loc[RULE]
    t.columns = [f"fold {c}" for c in t.columns]
    t["mean"] = t.mean(axis=1)
    t["folds above the rule"] = [int((by_fold.pivot(index="method", columns="fold", values="P@50").loc[m] > rule_row).sum()) for m in order]
    t["folds above slide only"] = [int((by_fold.pivot(index="method", columns="fold", values="P@50").loc[m]
                                        > by_fold.pivot(index="method", columns="fold", values="P@50").loc[SLIDE_ONLY]).sum()) for m in order]
    return t


p50_main = p50_table(by_fold, ORDER)
print(f"Precision@50 on each client fold, Mar→Apr test ({len(pool):,} pool pages). A/B/C/D = what the model was trained on")
display(p50_main.round(3))
mean_main = by_fold.groupby("method")[METRICS].mean().loc[ORDER]
mean_main["P@50 minus random"] = mean_main["P@50"] - mean_main.loc[RANDOM, "P@50"]
print("Mean over the 5 client folds")
mean_main.round(3)

PASS  design B reproduces the W05 P@50 on every fold for both models
Precision@50 on each client fold, Mar→Apr test (84,592 pool pages). A/B/C/D = what the model was trained on


,fold 0,fold 1,fold 2,fold 3,fold 4,mean,folds above the rule,folds above slide only
method,,,,,,,,
random order,0.405,0.708,0.467,0.407,0.53,0.503,0,1
W04 rule (baseline),0.800,0.920,0.880,0.700,0.62,0.784,0,2
slide only (no model),0.960,0.980,0.780,0.780,0.50,0.800,3,0
"logistic regression, A",0.980,0.980,0.900,0.840,0.70,0.880,5,4
"logistic regression, B",0.980,0.960,0.920,0.820,0.74,0.884,5,4
"logistic regression, C",0.940,0.940,0.760,0.680,0.42,0.748,2,0
"logistic regression, D",0.960,0.960,0.800,0.780,0.38,0.776,3,1
"gradient boosting, A",0.920,0.920,0.880,0.880,0.88,0.896,3,3
"gradient boosting, B",0.880,0.940,0.840,0.920,0.86,0.888,4,3


Mean over the 5 client folds


,P@20,P@50,P@100,avg precision,NDCG@50,ROC AUC,P@50 minus random
method,,,,,,,
random order,0.503,0.503,0.503,0.503,0.503,0.500,0.000
W04 rule (baseline),0.850,0.784,0.810,0.597,0.789,0.616,0.281
slide only (no model),0.780,0.800,0.840,0.686,0.787,0.684,0.297
"logistic regression, A",0.880,0.880,0.872,0.695,0.877,0.705,0.377
"logistic regression, B",0.900,0.884,0.864,0.692,0.888,0.701,0.381
"logistic regression, C",0.770,0.748,0.806,0.663,0.759,0.664,0.245
"logistic regression, D",0.800,0.776,0.802,0.661,0.778,0.661,0.273
"gradient boosting, A",0.910,0.896,0.896,0.721,0.897,0.733,0.393
"gradient boosting, B",0.890,0.888,0.906,0.699,0.880,0.709,0.385


**What the numbers say (March→April test).**

- **Grouping by client was not the weak spot.** With random rows (A) my logistic regression gets P@50 0.880 and ROC AUC 0.705, almost the same as the grouped W05 split (B: 0.884 and 0.701). With five page-level features and no client id it has little room to learn a client. Gradient boosting has more room and uses it: ROC AUC 0.733 and average precision 0.721 with random rows, 0.709 and 0.699 grouped. So a random split would have flattered the flexible model, not the simple one. This is my small version of question 2 for Finding B.
- **Time is the weak spot.** Trained only on what was known by 31 March, my model's P@50 drops from 0.884 (B) to 0.776 (D) and 0.748 (C). Average precision drops from 0.692 to 0.661 and ROC AUC from 0.701 to 0.661. C has more training pages than D and still does worse, so the drop is about the month, not about the amount of data.
- **With the honest split the W05 win over the rule is gone in this month.** Under D the model is above the rule on 3 of 5 folds and level on the mean (0.776 vs 0.784). It is above the slide-only sort on 1 of 5 folds (mean 0.776 vs 0.800). Over the whole queue it still ranks better than the rule (average precision 0.661 vs 0.597) but not better than the slide alone (0.686).
- **Fold 4 breaks.** Under D its P@50 is 0.38, below a random queue for that fold (0.53). The failure examples below show why.
- **Gradient boosting doesn't rescue it.** Time-aware it gets 0.864 (C) and 0.760 (D). Under D its folds go from 0.48 to 0.98, so the steadier folds W05 saw (0.84 to 0.94) don't hold.

### A second time step: train on March→April, test on April→May

One month pair can be a fluke, so I repeat the test one month later. The test pages are the Apr→May review pool (84,691 pages from 43 clients) in five client folds of its own. C and D train on the Mar→Apr pool, B on the Apr→May pages of the other clients.

In [10]:
# Second time step: test on Apr→May, train on Mar→Apr (C, D) or on Apr→May of other clients (B)
next_scores = {RULE: next_pool["rule_score"].to_numpy(), SLIDE_ONLY: -next_pool["late_half_share"].to_numpy()}
for code in ["B", "C", "D"]:
    for model_name, make_model in MODELS.items():
        next_scores[f"{model_name}, {code}"] = out_of_fold(code, make_model, next_pool, pool)
by_fold_next = evaluate(next_pool, next_scores)

ORDER_NEXT = [RANDOM, RULE, SLIDE_ONLY] + [f"{m}, {c}" for m in MODELS for c in ["B", "C", "D"]]
p50_next = p50_table(by_fold_next, ORDER_NEXT)
print(f"Apr→May test: {len(next_pool):,} pool pages from {next_pool['client_hash_id'].nunique()} clients, "
      f"base rate in the pool {next_pool[LABEL].mean():.3f}. C and D were trained on the Mar→Apr pool")
display(p50_next.round(3))
mean_next = by_fold_next.groupby("method")[METRICS].mean().loc[ORDER_NEXT]
mean_next["P@50 minus random"] = mean_next["P@50"] - mean_next.loc[RANDOM, "P@50"]
mean_next.round(3)

Apr→May test: 84,691 pool pages from 43 clients, base rate in the pool 0.573. C and D were trained on the Mar→Apr pool


,fold 0,fold 1,fold 2,fold 3,fold 4,mean,folds above the rule,folds above slide only
method,,,,,,,,
random order,0.397,0.591,0.595,0.631,0.691,0.581,0,0
W04 rule (baseline),0.740,0.960,0.940,0.860,0.920,0.884,0,0
slide only (no model),0.960,0.960,1.000,1.000,0.980,0.980,4,0
"logistic regression, B",0.940,0.940,1.000,0.960,0.980,0.964,4,0
"logistic regression, C",0.980,0.980,1.000,0.980,1.000,0.988,5,3
"logistic regression, D",0.980,0.980,1.000,0.980,1.000,0.988,5,3
"gradient boosting, B",1.000,0.960,0.940,1.000,0.980,0.976,3,1
"gradient boosting, C",1.000,0.980,1.000,1.000,0.980,0.992,5,2
"gradient boosting, D",0.880,1.000,0.980,0.960,0.960,0.956,5,1


,P@20,P@50,P@100,avg precision,NDCG@50,ROC AUC,P@50 minus random
method,,,,,,,
random order,0.581,0.581,0.581,0.581,0.581,0.500,0.000
W04 rule (baseline),0.890,0.884,0.880,0.713,0.890,0.666,0.303
slide only (no model),1.000,0.980,0.974,0.771,0.984,0.688,0.399
"logistic regression, B",0.980,0.964,0.944,0.772,0.971,0.709,0.383
"logistic regression, C",1.000,0.988,0.968,0.765,0.991,0.694,0.407
"logistic regression, D",1.000,0.988,0.970,0.763,0.991,0.693,0.407
"gradient boosting, B",1.000,0.976,0.970,0.799,0.982,0.739,0.395
"gradient boosting, C",0.990,0.992,0.978,0.781,0.991,0.699,0.411
"gradient boosting, D",0.960,0.956,0.966,0.779,0.963,0.697,0.375


**What the second step says.** Here the time-aware model holds up: P@50 0.988 under both C and D, against 0.964 for the grouped same-month design (B), 0.884 for the rule and 0.581 for a random queue. It is above the rule on all 5 folds. But a plain sort by the late-April slide gets 0.980, so the model again only matches the slide-only sort (above it on 3 of 5 folds, average precision 0.763 vs 0.771). Gradient boosting under D gets 0.956, again not better than the logistic regression.

So the time-aware drop showed up in one of the two month pairs. It is not a steady loss. What repeats in both pairs is this: **once the model is trained only on the past, it is not clearly better than sorting by the slide alone.**

### Before / after in one table (my model)

In [11]:
# Before / after in one table: my model (logistic regression) next to the two no-training floors
def row(mean_table, p50_table_, method):
    return {"P@50": mean_table.loc[method, "P@50"], "P@50 lowest fold": p50_table_.loc[method].iloc[:N_FOLDS].min(),
            "avg precision": mean_table.loc[method, "avg precision"], "ROC AUC": mean_table.loc[method, "ROC AUC"],
            "folds above the rule": p50_table_.loc[method, "folds above the rule"],
            "folds above slide only": p50_table_.loc[method, "folds above slide only"]}

before_after = pd.DataFrame({
    ("Mar→Apr", "A. random rows (reference)"): row(mean_main, p50_main, f"{MAIN}, A"),
    ("Mar→Apr", "B. grouped (W05, before)"): row(mean_main, p50_main, f"{MAIN}, B"),
    ("Mar→Apr", "C. time-aware (after)"): row(mean_main, p50_main, f"{MAIN}, C"),
    ("Mar→Apr", "D. time-aware + grouped (after)"): row(mean_main, p50_main, f"{MAIN}, D"),
    ("Mar→Apr", "W04 rule (no training)"): row(mean_main, p50_main, RULE),
    ("Mar→Apr", "slide only (no training)"): row(mean_main, p50_main, SLIDE_ONLY),
    ("Apr→May", "B. grouped (before)"): row(mean_next, p50_next, f"{MAIN}, B"),
    ("Apr→May", "D. time-aware + grouped (after)"): row(mean_next, p50_next, f"{MAIN}, D"),
    ("Apr→May", "W04 rule (no training)"): row(mean_next, p50_next, RULE),
    ("Apr→May", "slide only (no training)"): row(mean_next, p50_next, SLIDE_ONLY),
}).T
before_after = before_after.astype({"folds above the rule": int, "folds above slide only": int})
before_after.index.names = ["test month", f"{MAIN} under split"]
print(f"Base rate in the pool: Mar→Apr {pool[LABEL].mean():.3f}, Apr→May {next_pool[LABEL].mean():.3f} (= P@50 of a random queue)")
before_after.round(3)

Base rate in the pool: Mar→Apr 0.497, Apr→May 0.573 (= P@50 of a random queue)


P@50  P@50 lowest fold  \
test month logistic regression under split                            
Mar→Apr    A. random rows (reference)       0.880              0.70   
           B. grouped (W05, before)         0.884              0.74   
           C. time-aware (after)            0.748              0.42   
           D. time-aware + grouped (after)  0.776              0.38   
           W04 rule (no training)           0.784              0.62   
           slide only (no training)         0.800              0.50   
Apr→May    B. grouped (before)              0.964              0.94   
           D. time-aware + grouped (after)  0.988              0.98   
           W04 rule (no training)           0.884              0.74   
           slide only (no training)         0.980              0.96   

                                            avg precision  ROC AUC  \
test month logistic regression under split                           
Mar→Apr    A. random rows (reference)               0.695    0.705   
           B. grouped (W05, before)                 0.692    0.701   
           C. time-aware (after)                    0.663    0.664   
           D. time-aware + grouped (after)          0.661    0.661   
           W04 rule (no training)                   0.597    0.616   
           slide only (no training)                 0.686    0.684   
Apr→May    B. grouped (before)                      0.772    0.709   
           D. time-aware + grouped (after)          0.763    0.693   
           W04 rule (no training)                   0.713    0.666   
           slide only (no training)                 0.771    0.688   

                                            folds above the rule  \
test month logistic regression under split                         
Mar→Apr    A. random rows (reference)                          5   
           B. grouped (W05, before)                            5   
           C. time-aware (after)                               2   
           D. time-aware + grouped (after)                     3   
           W04 rule (no training)                              0   
           slide only (no training)                            3   
Apr→May    B. grouped (before)                                 4   
           D. time-aware + grouped (after)                     5   
           W04 rule (no training)                              0   
           slide only (no training)                            4   

                                            folds above slide only  
test month logistic regression under split                          
Mar→Apr    A. random rows (reference)                            4  
           B. grouped (W05, before)                              4  
           C. time-aware (after)                                 0  
           D. time-aware + grouped (after)                       1  
           W04 rule (no training)                                2  
           slide only (no training)                              0  
Apr→May    B. grouped (before)                                   0  
           D. time-aware + grouped (after)                       3  
           W04 rule (no training)                                0  
           slide only (no training)                              0

**My honest numbers from now on are the time-aware + grouped ones (D):** P@50 0.776 on March→April and 0.988 on April→May, next to the rule (0.784 and 0.884), the slide-only sort (0.800 and 0.980) and a random queue (0.503 and 0.581). The W05 number (0.884) was honest about clients but not about time.

### Why the March→April step dropped: the weights move between months

In [12]:
# Same logistic regression fitted on each slice's whole pool: do the weights stay put from month to month?
weights, single_auc = {}, {}
for name, d in [("Feb→Mar", prev_pool), ("Mar→Apr", pool), ("Apr→May", next_pool)]:
    weights[name] = logistic_regression().fit(d[FEATURES], d[LABEL])[-1].coef_[0]
    single_auc[name] = [roc_auc_score(d[LABEL], d[f]) for f in FEATURES]
months_table = pd.concat({"LR weight (per 1 std)": pd.DataFrame(weights, index=FEATURES),
                          "ROC AUC of the feature alone": pd.DataFrame(single_auc, index=FEATURES)}, axis=1)
months_table.index.name = "feature"
print("Negative weight / AUC under 0.5 = higher value, LESS likely to decline")
months_table.round(3)

Negative weight / AUC under 0.5 = higher value, LESS likely to decline


LR weight (per 1 std)                  \
                              Feb→Mar Mar→Apr Apr→May   
feature                                                 
imp_per_day_log                -0.065  -0.200   0.128   
ctr_pct                        -0.224  -0.382  -0.349   
avg_position                   -0.294  -0.102   0.101   
late_half_share                -0.362  -0.691  -0.778   
gsc_days                       -0.126  -0.005  -0.241   

                ROC AUC of the feature alone                  
                                     Feb→Mar Mar→Apr Apr→May  
feature                                                       
imp_per_day_log                        0.489   0.473   0.518  
ctr_pct                                0.445   0.405   0.412  
avg_position                           0.450   0.492   0.499  
late_half_share                        0.415   0.324   0.321  
gsc_days                               0.500   0.571   0.504

The same model fitted on three different months gives different weights. Only **CTR** and the **slide** (`late_half_share`) keep a clear weight with the same sign in all three months. They are also the two signals the W04 audit marked CONFIRMED (`gsc_days` stays negative too, but it is almost zero in Mar→Apr). Volume and position change sign: position weighs −0.29 in Feb→Mar but +0.10 in Apr→May, volume −0.20 in Mar→Apr but +0.13 in Apr→May. This fits the MIXED verdict for position in W04, where page-one pages declined more often in Feb→Mar but not in Mar→Apr.

The Feb→Mar model (the one C and D use for the main test) leans on the slide about half as much as the Mar→Apr model (−0.36 vs −0.69) and three times more on position (−0.29 vs −0.10). The Mar→Apr model used in the second step had CTR and slide weights close to the next month's own (−0.38 vs −0.35 and −0.69 vs −0.78), which fits with that step holding up. With two steps I can't say which month is "normal". I can only say that the weights other than CTR and the slide are not stable from month to month.

### Real failure examples under the honest split

All under design D, my strictest honest split, on the March→April test. First what the top 50 of each fold looks like, then who the false alarms and the misses are, then three single pages. "Young page" = the page does not have rows on all 28 days of February (a new page or a newly tracked one). April numbers are hindsight and are joined only now, after every score is fixed.

In [13]:
# Real failure examples, under the strictest honest design (D) for my model
D_MAIN, B_MAIN = f"{MAIN}, D", f"{MAIN}, B"
hind = HINDSIGHT["Mar→Apr"]
pool = pool.join(hind, on="content_hash_id")       # hindsight joined only now, after every score is fixed
feb = MONTHS["2026-02"].set_index("content_hash_id")
pool["feb_days"] = feb["n_rows"].reindex(pool["content_hash_id"]).fillna(0).to_numpy()   # days with a row in February
pool["young"] = pool["feb_days"] < 28                                                    # not a full February: a new or newly tracked page
client_pages = main.groupby("client_hash_id").size().reset_index(name="n").sort_values(["n", "client_hash_id"], ascending=[False, True])
pool["client_rank"] = pool["client_hash_id"].map(pd.Series(np.arange(1, len(client_pages) + 1), index=client_pages["client_hash_id"].to_numpy()))


def top_k_per_fold(score, k=K_MAIN):
    parts = []
    for fold in range(N_FOLDS):
        part = pool[pool["fold"] == fold]
        parts.append(part.iloc[rank_order(np.asarray(score)[part.index], part["content_hash_id"])[:k]])
    return pd.concat(parts)


def top_profile(top):
    return pd.Series({
        "declined (precision)": top[LABEL].mean(),
        "median avg position": top["avg_position"].median(),
        "median slide % (early → late March)": top["slide_pct"].median(),
        "median impressions per day": top["imp_per_day"].median(),
        "young page (not a full February)": top["young"].mean(),
        "back above its March level in April (hindsight)": (top["label_vs_month_pct"] > 0).mean(),
        "biggest single-client share of the 50": top["client_rank"].value_counts().iloc[0] / len(top),
    })


top_d, top_b = top_k_per_fold(scores[D_MAIN]), top_k_per_fold(scores[B_MAIN])
profile = pd.concat({f"fold {k}": top_profile(top_d[top_d["fold"] == k]) for k in range(N_FOLDS)}, axis=1)
profile["all 250 (D)"] = top_profile(top_d)
profile["all 250 (B, W05)"] = top_profile(top_b)
print(f"Top {K_MAIN} of each fold's queue, {MAIN}. Columns fold 0 to 4 are design D")
profile.round(3)

Top 50 of each fold's queue, logistic regression. Columns fold 0 to 4 are design D


,fold 0,fold 1,fold 2,fold 3,fold 4,all 250 (D),"all 250 (B, W05)"
declined (precision),0.960,0.960,0.800,0.780,0.380,0.776,0.884
median avg position,3.776,4.366,5.491,5.635,4.384,4.767,5.945
median slide % (early → late March),86.407,90.762,77.847,79.540,76.027,83.505,85.317
median impressions per day,34.629,45.081,32.839,27.065,19.952,31.790,32.145
young page (not a full February),0.220,0.360,0.860,0.580,0.600,0.524,0.452
back above its March level in April (hindsight),0.040,0.040,0.180,0.160,0.540,0.192,0.088
biggest single-client share of the 50,1.000,0.920,0.380,0.420,0.520,0.200,0.200


Under D folds 0 and 1 still look like W05 (0.96 precision). Folds 2 to 4 are where it goes wrong and they share a pattern: more **young pages** (86%, 58% and 60% of their top 50) and more pages that were **back above their March level in April** (18%, 16% and 54%). Fold 4 is the extreme, where more than half of the top 50 came back. Compared with the W05 top 50 (design B), the D top 50 has pages ranked higher in search (median position 4.8 vs 5.9), more young pages (52% vs 45%) and twice the share that came back (19% vs 9%).

In [14]:
# Who are the false alarms and who are the misses? (design D)
hits, alarms = top_d[top_d[LABEL] == 1], top_d[top_d[LABEL] == 0]
alarm_table = pd.DataFrame({name: {
    "pages": len(g),
    "young page (not a full February)": g["young"].mean(),
    "median late-month impressions per day": g["late_per_day"].median(),
    "median slide %": g["slide_pct"].median(),
    "share with 0 clicks in March": (g["clicks_per_day"] == 0).mean(),
    "median April vs March % (hindsight)": g["label_vs_month_pct"].median(),
    "back above its March level in April (hindsight)": (g["label_vs_month_pct"] > 0).mean(),
} for name, g in [("false alarms in the top 50s", alarms), ("hits in the top 50s", hits)]})
display(alarm_table.round(3))

# Fold 4 is where design D fails hardest: which client fills its top 50?
fold4 = top_d[top_d["fold"] == 4]
big_rank = fold4["client_rank"].value_counts().index[0]
big = pool[pool["client_rank"] == big_rank]
rest4 = pool[(pool["fold"] == 4) & (pool["client_rank"] != big_rank)]
print(f"Fold 4: client rank {big_rank} fills {int((fold4['client_rank'] == big_rank).sum())} of its top {K_MAIN} under D "
      f"({int((top_b[top_b['fold'] == 4]['client_rank'] == big_rank).sum())} under B)")
display(pd.DataFrame({f"client rank {big_rank}: all its pool pages": {
                          "pages": len(big), "decline rate": big[LABEL].mean(), "share that slid 20%+ in late March": (big["slide_pct"] >= 20).mean(),
                          "median April vs March % (hindsight)": big["label_vs_month_pct"].median(), "young page share": big["young"].mean()},
                      f"client rank {big_rank}: its pages in the D top 50": {
                          "pages": int((fold4["client_rank"] == big_rank).sum()), "decline rate": fold4.loc[fold4["client_rank"] == big_rank, LABEL].mean(),
                          "share that slid 20%+ in late March": (fold4.loc[fold4["client_rank"] == big_rank, "slide_pct"] >= 20).mean(),
                          "median April vs March % (hindsight)": fold4.loc[fold4["client_rank"] == big_rank, "label_vs_month_pct"].median(),
                          "young page share": fold4.loc[fold4["client_rank"] == big_rank, "young"].mean()},
                      "the other clients of fold 4": {
                          "pages": len(rest4), "decline rate": rest4[LABEL].mean(), "share that slid 20%+ in late March": (rest4["slide_pct"] >= 20).mean(),
                          "median April vs March % (hindsight)": rest4["label_vs_month_pct"].median(), "young page share": rest4["young"].mean()}}).round(3))

# The other side: declined pages ranked in the bottom half of their fold
pct_rank = pd.Series(scores[D_MAIN]).groupby(pool["fold"]).rank(pct=True)
missed = pool[(pool[LABEL] == 1) & (pct_rank < 0.5)]
print(f"Declined pages ranked in the bottom half under D: {len(missed):,} of {int(pool[LABEL].sum()):,} ({len(missed) / pool[LABEL].sum():.1%}). "
      f"{(missed['slide_pct'] < 0).mean():.1%} of them ROSE in late March (median slide {missed['slide_pct'].median():.1f}%)")

,false alarms in the top 50s,hits in the top 50s
pages,56.000,194.000
young page (not a full February),0.714,0.469
median late-month impressions per day,7.562,8.250
median slide %,79.007,84.873
share with 0 clicks in March,0.268,0.536
median April vs March % (hindsight),112.117,-85.428
back above its March level in April (hindsight),0.857,0.000


Fold 4: client rank 20 fills 26 of its top 50 under D (17 under B)


,client rank 20: all its pool pages,client rank 20: its pages in the D top 50,the other clients of fold 4
pages,259.000,26.000,15611.000
decline rate,0.274,0.154,0.534
share that slid 20%+ in late March,0.483,1.000,0.143
median April vs March % (hindsight),62.559,99.897,-24.978
young page share,1.000,1.000,0.075


Declined pages ranked in the bottom half under D: 16,091 of 42,071 (38.2%). 73.8% of them ROSE in late March (median slide -29.7%)


**Three kinds of failure:**

1. **Young pages with a short dip (most false alarms).** The five top 50s hold 56 false alarms (W05 had 29). They are mostly young (71% vs 47% of the hits) and 86% of them ended April above their March level (median +112%). They had slid about as hard as the hits (median slide 79% vs 85%), so nothing in March told them apart.
2. **One young, growing client fills fold 4.** Client rank 20 has 259 pool pages and all of them are young. 48% of them slid 20%+ in late March, yet the median page ended April 63% *above* March and only 27% declined. Under D this client takes 26 of the fold's top 50 (17 under B) and only 4 of those 26 declined. The weekly table in the next cell shows why: the client as a whole was growing fast (its average page went from about 50 impressions a day in early March to 150 to 200 in April), while single pages swung up and down from week to week. Scoring pages one at a time, the model reads each swing down as a decline. This is the W03 limitation again: part of the label moves with the client and my five page features can't see the client.
3. **Pages that grew in March (the misses).** 16,091 of the 42,071 declined pool pages (38.2%) sit in the bottom half of their fold's queue. 73.8% of them rose in late March. They fell in April after growing, often a surge ending. The model reads "rising" as safe.

In [15]:
# Three concrete wrong cases under D, plus the weekly story of the fold-4 client. Hash ids stay out of every display
pool["fold_rank"] = pd.Series(scores[D_MAIN]).groupby(pool["fold"]).rank(ascending=False, method="first").astype(int)
top_d = top_d.join(pool["fold_rank"])
case_1 = top_d[top_d[LABEL] == 0].sort_values(["fold_rank", "content_hash_id"]).index[0]                 # highest-ranked false alarm
case_2 = top_d[(top_d[LABEL] == 0) & (top_d.index != case_1)].sort_values("imp_per_day", ascending=False).index[0]  # biggest false alarm
missed = missed.assign(lost_per_day=missed["imp_per_day"] - missed["label_per_day"])
case_3 = missed.sort_values("lost_per_day", ascending=False).index[0]                                    # biggest loss ranked low
cases = pool.loc[[case_1, case_2, case_3]].copy()
cases["model score (D)"] = scores[D_MAIN][cases.index]
cases.index = ["case 1: highest-ranked false alarm", "case 2: biggest false alarm in a top 50", "case 3: biggest missed decline"]
case_cols = ["fold", "client_rank", "fold_rank", "model score (D)", "feb_days", "early_per_day", "late_per_day", "slide_pct",
             "clicks_per_day", "ctr_pct", "avg_position", "label_per_day", "label_vs_month_pct", LABEL]
display(cases[case_cols].rename(columns={"label_per_day": "april_per_day (hindsight)", "label_vs_month_pct": "april_vs_march_% (hindsight)",
                                         LABEL: f"{LABEL} (hindsight)"}).T.round(3))

# Weekly average impressions per day, February to April (April is hindsight, read only now)
groups = pd.DataFrame({"content_hash_id": pool.loc[[case_1, case_2, case_3], "content_hash_id"].to_numpy(),
                       "series": ["case 1", "case 2", "case 3"]})
groups = pd.concat([groups, pd.DataFrame({"content_hash_id": big["content_hash_id"].to_numpy(),
                                          "series": f"client rank {big_rank}, all {len(big)} pool pages (mean per page)"})])
con.register("series_pages", groups)
daily = con.sql(f'''
    SELECT s.series, f.report_date, AVG(f.gsc_impressions) AS impressions
    FROM read_parquet(['{REL}/fact_content_daily_performance/month=2026-02/*.parquet',
                       '{REL}/fact_content_daily_performance/month=2026-03/*.parquet',
                       '{REL}/fact_content_daily_performance/month=2026-04/*.parquet']) f
    JOIN series_pages s USING (content_hash_id)
    GROUP BY s.series, f.report_date
''').df()
daily["report_date"] = pd.to_datetime(daily["report_date"])
daily["week"] = daily["report_date"].dt.to_period("W-SUN").dt.start_time
weekly = daily.groupby(["series", "week"])["impressions"].mean().unstack("week").sort_index()
weekly.columns = [c.strftime("%d %b") for c in weekly.columns]
print("Average impressions per day, by week starting (February to April; April is hindsight)")
with pd.option_context("display.max_columns", 20):
    display(weekly.round(0))

,case 1: highest-ranked false alarm,case 2: biggest false alarm in a top 50,case 3: biggest missed decline
fold,1.000,2.000,2.000
client_rank,16.000,8.000,3.000
fold_rank,1.000,14.000,15398.000
model score (D),0.705,0.511,0.136
feb_days,6.000,10.000,28.000
early_per_day,30.267,517.667,481.533
late_per_day,5.562,47.188,12823.812
slide_pct,81.622,90.885,-2563.120
clicks_per_day,0.000,0.355,0.774
ctr_pct,0.000,0.129,0.011


Average impressions per day, by week starting (February to April; April is hindsight)


,26 Jan,02 Feb,09 Feb,16 Feb,23 Feb,02 Mar,09 Mar,16 Mar,23 Mar,30 Mar,06 Apr,13 Apr,20 Apr,27 Apr
series,,,,,,,,,,,,,,
case 1,NaN,NaN,NaN,NaN,0.0,30.0,35.0,12.0,1.0,0.0,0.0,0.0,42.0,71.0
case 2,NaN,NaN,NaN,0.0,85.0,575.0,492.0,87.0,19.0,701.0,1748.0,1857.0,1010.0,546.0
case 3,50.0,44.0,621.0,11669.0,589.0,307.0,683.0,300.0,28352.0,4017.0,2701.0,616.0,359.0,278.0
"client rank 20, all 259 pool pages (mean per page)",NaN,NaN,NaN,9.0,49.0,54.0,48.0,56.0,79.0,140.0,197.0,202.0,153.0,162.0


| Case | What the model saw (up to 31 March) | What happened in April (hindsight) | Why it is hard |
|---|---|---|---|
| **1. #1 in fold 1** (score 0.71, client rank 16) | A young page (rows on only 6 February days). 30 impressions a day in early March, 5.6 in late March, no clicks, position 3.6 | Close to zero from the week of 23 March through the week of 13 April, then back in the last two weeks of April (42 and 71 a day). The April average (19.2 a day) ended 9.8% above March, so the label says "not declined" | As much a label miss as a model miss. A reviewer would want to see a page that vanished for about four weeks, but the monthly average hides it |
| **2. #14 in fold 2** (score 0.51, client rank 8) | Young (rows on 10 February days). 518 a day in early March at position 2.2, then 47 a day in late March (−91%) | Back from the week of 30 March, then 1,313 a day in April (+378%) | Same page as W05 case 2. A short dip on a new page looks exactly like a real drop inside March |
| **3. #15,398 of 16,853 in fold 2** (score 0.14, client rank 3) | Rose from 482 to 12,824 a day (the week of 23 March averaged about 28,000 a day). CTR 0.011% and an odd position of 0.67 | Fell back to 1,678 a day in April (−75.5% vs the March average) | Same page as W05 case 3. It lives on spikes: February had one too (the week of 16 February, about 11,700 a day). The "decline" is the spike ending, which a refresh can't prevent |

Cases 2 and 3 are the same pages W05 found under the grouped split, so these errors do not come from the split. They are limits of what one month of page data can show.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The same hunt as W03, now on the final feature set (the five W03 features in the logistic regression) and with the attack checklist from the `hunting-leakage-and-validating` skill. Five parts: (a) the timeline, (b) a leak trap with window-overlap features on purpose, (c) training without each feature, (d) the label carry-over and (e) the checklist filled in.

### 3a. Timeline: is every input known at the decision moment?

In [16]:
# 3a. Timeline: every input, the dates it reads and whether it is known at the decision moment
timeline = pd.DataFrame([
    ("5 features + W04 rule + review pool (main test)", "2026-03", MONTHS["2026-03"]["first_date"].min(), MONTHS["2026-03"]["last_date"].max()),
    ("label of the main test", "2026-04", MONTHS["2026-04"]["first_date"].min(), MONTHS["2026-04"]["last_date"].max()),
    ("features of the time-aware training slice (C, D)", "2026-02", MONTHS["2026-02"]["first_date"].min(), MONTHS["2026-02"]["last_date"].max()),
    ("labels of the time-aware training slice (C, D)", "2026-03", MONTHS["2026-03"]["first_date"].min(), MONTHS["2026-03"]["last_date"].max()),
], columns=["input", "month partition", "first day read", "last day read"])
timeline["known by end of 31 March?"] = np.where(pd.to_datetime(timeline["last day read"]) <= DECISION_MOMENT, "yes", "NO (label only)")
display(timeline)
assert pd.to_datetime(MONTHS["2026-03"]["last_date"].max()) == DECISION_MOMENT < pd.to_datetime(MONTHS["2026-04"]["first_date"].min())

# Eligibility rule 4 ("a row on every day of the label month") reads the label month. Does it drop anyone?
print("Pages removed ONLY by rule 4 (rules 1 to 3 passed):", RULE_4_REMOVED)

# Feature list: no ids, no label, no label-derived column, no FlyRank flag or score
NEVER = {"content_hash_id", "client_hash_id", LABEL, STRICT_LABEL, "label_per_day", "label_vs_month_pct", "label_vs_late_half_pct",
         "rule_score", "fold", "trend_pct", "trend_direction", "health_score", "optimization_flags"}
assert not set(FEATURES) & NEVER
print("PASS  no ids, labels, label-derived columns, rule scores or FlyRank flags among the features:", FEATURES)

,input,month partition,first day read,last day read,known by end of 31 March?
0,5 features + W04 rule + review pool (main test),2026-03,2026-03-01,2026-03-31,yes
1,label of the main test,2026-04,2026-04-01,2026-04-30,NO (label only)
2,"features of the time-aware training slice (C, D)",2026-02,2026-02-01,2026-02-28,yes
3,"labels of the time-aware training slice (C, D)",2026-03,2026-03-01,2026-03-31,yes


Pages removed ONLY by rule 4 (rules 1 to 3 passed): {'Feb→Mar': 7, 'Mar→Apr': 0, 'Apr→May': 0}
PASS  no ids, labels, label-derived columns, rule scores or FlyRank flags among the features: ['imp_per_day_log', 'ctr_pct', 'avg_position', 'late_half_share', 'gsc_days']


Every feature of the main test reads 1 to 31 March and the label starts on 1 April. The time-aware training slice (C, D) reads February for its features and March for its labels, all complete by the end of 31 March.

**Filters can leak too.** W03 rule 4 ("a row on every day of the label month") looks at the label month. On 31 March nobody knows whether a page will have all 30 April rows, so in theory this filter picks pages with the future. In practice it removes **0** pages on its own in March→April and in April→May: every page that passed rules 1 to 3 also had a full label month. In Feb→Mar it removes 7 pages, but that slice's label month is March, which is complete by 31 March. The review pool reads only late-month numbers. No feature is an id, a label, a label-derived column, a rule score or a FlyRank flag.

### 3b. The trap: window-overlap features on purpose

In W03 I added the label-derived column and the score jumped to 1.000. This time I test the quieter kind from Finding B: features whose **window overlaps the label window**. Each one is added alone to my model under the W05 split, then all three are deleted:

- `april_first_week_log`: impressions per day on 1 to 7 April (inside the label window)
- `impressions_3_months_log`: impressions per day over February to April, like a 90-day column next to a 30-day trend label
- `days_visible_3_months`: days with 1+ impression over February to April, like the paper's "days visible in 90"

In [17]:
# 3b. The trap: window-overlap features ON PURPOSE, then removed. Same model, same W05 split (design B)
apr, feb = MONTHS["2026-04"].set_index("content_hash_id"), MONTHS["2026-02"].set_index("content_hash_id")
mar = MONTHS["2026-03"].set_index("content_hash_id")
ids = pool["content_hash_id"]
TRAPS = {
    "april_first_week_log": np.log1p(apr["imp_first_7_days"].reindex(ids).to_numpy() / 7),              # inside the label window
    "impressions_3_months_log": np.log1p((feb["imp"].reindex(ids).fillna(0).to_numpy() + mar["imp"].reindex(ids).to_numpy()
                                          + apr["imp"].reindex(ids).to_numpy()) / 89),                      # like a 90-day window
    "days_visible_3_months": (feb["days_with_impressions"].reindex(ids).fillna(0).to_numpy()
                              + mar["days_with_impressions"].reindex(ids).to_numpy()
                              + apr["days_with_impressions"].reindex(ids).to_numpy()),                      # like "days visible in 90"
}


def grouped_scores(columns, label=LABEL, data=None):
    data = pool if data is None else data
    s = np.full(len(data), np.nan)
    for fold in range(N_FOLDS):
        train, test = data["fold"] != fold, data["fold"] == fold
        s[test.to_numpy()] = logistic_regression().fit(data.loc[train, columns], data.loc[train, label]).predict_proba(data.loc[test, columns])[:, 1]
    return s


def mean_metrics(s, label=LABEL):
    m = evaluate(pool, {"x": s}, label)
    return m[m["method"] == "x"][["P@50", "avg precision", "ROC AUC"]].mean()


trap_pool = pool.assign(**TRAPS)
trap_rows = {"1. honest: the 5 features": mean_metrics(grouped_scores(FEATURES, data=trap_pool))}
for i, (col, label_) in enumerate([("april_first_week_log", "2. + April 1-7 impressions (inside the label window)"),
                                    ("impressions_3_months_log", "3. + Feb-Apr impressions per day (window overlaps the label)"),
                                    ("days_visible_3_months", "4. + days visible Feb-Apr (window overlaps the label)")]):
    trap_rows[label_] = mean_metrics(grouped_scores(FEATURES + [col], data=trap_pool))
trap_pool = trap_pool.drop(columns=list(TRAPS))                  # traps deleted
assert not set(TRAPS) & set(trap_pool.columns)
trap_rows["5. traps deleted: the 5 features again"] = mean_metrics(grouped_scores(FEATURES, data=trap_pool))
trap_table = pd.DataFrame(trap_rows).T
trap_table["random order (base rate)"] = pool.groupby("fold")[LABEL].mean().mean()
trap_table.round(3)

,P@50,avg precision,ROC AUC,random order (base rate)
1. honest: the 5 features,0.884,0.692,0.701,0.503
2. + April 1-7 impressions (inside the label window),0.960,0.889,0.899,0.503
3. + Feb-Apr impressions per day (window overlaps the label),1.000,0.886,0.885,0.503
4. + days visible Feb-Apr (window overlaps the label),0.932,0.694,0.705,0.503
5. traps deleted: the 5 features again,0.884,0.692,0.701,0.503


**The harness works and the overlap is not small.** One week of April lifts P@50 from 0.884 to 0.960 and ROC AUC from 0.701 to 0.899. Three months of impressions that include April give a perfect top 50 (P@50 1.000, ROC AUC 0.885). Days visible is the mildest: average precision and ROC AUC barely move (0.694 and 0.705), but P@50 still rises to 0.932, because pages that went dark in April have fewer visible days and float to the top. After the traps are deleted the honest numbers come back exactly (0.884, 0.692 and 0.701).

**For Finding B** this does not prove that the paper's growth model leaked, since I don't know its exact windows. It shows that on this kind of data a "90-day impressions" feature next to a "last 30 vs previous 30 days" label can make a model look much better than it is. That is why my first question to the paper is about the windows.

### 3c. Each feature removed and each feature alone

In [18]:
# 3c. Train without each feature (same W05 split), plus each feature alone on the held-out clients
honest = mean_metrics(scores[B_MAIN])
rows = {"all 5 features": honest}
for f in FEATURES:
    rows[f"without {f}"] = mean_metrics(grouped_scores([c for c in FEATURES if c != f]))
drop_table = pd.DataFrame(rows).T
drop_table["P@50 change"] = drop_table["P@50"] - honest["P@50"]
display(drop_table.round(3))

alone = pd.DataFrame({f: {"ROC AUC alone, mean of held-out folds": np.mean([roc_auc_score(g[LABEL], g[f]) for _, g in pool.groupby("fold")]),
                          "strength (distance from 0.5)": abs(np.mean([roc_auc_score(g[LABEL], g[f]) for _, g in pool.groupby("fold")]) - 0.5)}
                      for f in FEATURES}).T.sort_values("strength (distance from 0.5)", ascending=False)
alone.round(3)

,P@50,avg precision,ROC AUC,P@50 change
all 5 features,0.884,0.692,0.701,0.000
without imp_per_day_log,0.844,0.698,0.701,-0.040
without ctr_pct,0.796,0.676,0.684,-0.088
without avg_position,0.884,0.691,0.701,0.000
without late_half_share,0.652,0.598,0.626,-0.232
without gsc_days,0.880,0.693,0.703,-0.004


,"ROC AUC alone, mean of held-out folds",strength (distance from 0.5)
late_half_share,0.316,0.184
ctr_pct,0.402,0.098
gsc_days,0.556,0.056
imp_per_day_log,0.475,0.025
avg_position,0.513,0.013


**No feature is "too good".** The slide (`late_half_share`) carries the model: without it P@50 falls from 0.884 to 0.652 and ROC AUC from 0.701 to 0.626. That is a real drop, but not the collapse from about 1.0 that a leaked column gives (W03: 1.000 with the leak). On its own the slide gets ROC AUC 0.316, which is 0.684 read the other way (more slide, more decline), the same number as in W05. CTR is second (P@50 −0.088 without it) and volume third (−0.040). Position and `gsc_days` add nothing: dropping either one changes P@50 by 0.004 or less. None of the five comes close to perfect on its own.

### 3d. Label carry-over: the label shares March with the features

This is not a leak in the strict sense, because nothing after 31 March goes into the model. But the label compares April with the **whole March average**. That average includes late March, which the slide feature also sees. A page that only holds its late-March level counts as "declined" once its slide is above 34.1% (W04). So part of the label is already decided on 31 March. I measure how much of my model's success is this carry-over, then re-score everything on a stricter label that only counts a fall **after** the decision moment:

`fell_after_decision` = April impressions per day more than 20% below the **late-March** level.

In [19]:
# 3d. Label carry-over: the label's baseline (the March average) contains late March, which the features also see
EARLY_DAYS, LATE_DAYS, MARCH_DAYS = 15, 16, 31
r = 0.8 * EARLY_DAYS / (MARCH_DAYS - 0.8 * LATE_DAYS)
print(f"A page that just holds its late-March level through April is labelled 'declined' once its slide is above {100 * (1 - r):.1f}% (W04)")

carry_only = lambda d: (d[LABEL] == 1) & (d["label_vs_late_half_pct"] >= DECLINE_CUTOFF_PCT)   # declined vs March, but NOT below late March
carry = {"all declined pages in the pool": carry_only(pool).sum() / pool[LABEL].sum()}
for name, s in [(f"{MAIN}, B (W05)", scores[B_MAIN]), (f"{MAIN}, D", scores[D_MAIN]), (RULE, scores[RULE]), (SLIDE_ONLY, scores[SLIDE_ONLY])]:
    top = top_k_per_fold(s)
    carry[f"hits in the top 50s: {name}"] = carry_only(top).sum() / top[LABEL].sum()
display(pd.Series(carry, name="share of 'declined' pages that did NOT fall another 20% after 31 March").to_frame().round(3))

# The stricter label: April more than 20% below the LATE-March level, so only a fall after the decision moment counts
strict = evaluate(pool, {name: scores[name] for name in [RULE, SLIDE_ONLY, B_MAIN, D_MAIN, f"{CHECK}, B", f"{CHECK}, D"]}, STRICT_LABEL)
strict_scores_trained = grouped_scores(FEATURES, label=STRICT_LABEL)          # same model and split, trained FOR the stricter label
strict = pd.concat([strict, evaluate(pool, {f"{MAIN}, B, trained on the stricter label": strict_scores_trained}, STRICT_LABEL)
                    .query("method != @RANDOM")])
strict_table = strict.groupby("method")[["P@50", "avg precision", "ROC AUC"]].mean()
strict_table = strict_table.loc[[RANDOM, RULE, SLIDE_ONLY, B_MAIN, D_MAIN, f"{CHECK}, B", f"{CHECK}, D", f"{MAIN}, B, trained on the stricter label"]]
print(f"Stricter label '{STRICT_LABEL}': base rate in the pool {pool[STRICT_LABEL].mean():.3f}. Same pages, same folds")
display(strict_table.round(3))

top_strict = top_k_per_fold(strict_scores_trained)
print(f"Top 50s of the model trained on the stricter label: {(top_strict['slide_pct'] < 0).mean():.0%} ROSE in late March "
      f"(median slide {top_strict['slide_pct'].median():.0f}%), {(top_strict['clicks_per_day'] == 0).mean():.0%} had 0 clicks in March, "
      f"median {top_strict['imp_per_day'].median():.0f} impressions per day")

A page that just holds its late-March level through April is labelled 'declined' once its slide is above 34.1% (W04)


,share of 'declined' pages that did NOT fall another 20% after 31 March
all declined pages in the pool,0.134
"hits in the top 50s: logistic regression, B (W05)",0.430
"hits in the top 50s: logistic regression, D",0.361
hits in the top 50s: W04 rule (baseline),0.454
hits in the top 50s: slide only (no model),0.515


Stricter label 'fell_after_decision': base rate in the pool 0.501. Same pages, same folds


,P@50,avg precision,ROC AUC
method,,,
random order,0.508,0.508,0.500
W04 rule (baseline),0.428,0.502,0.492
slide only (no model),0.388,0.510,0.498
"logistic regression, B",0.504,0.541,0.549
"logistic regression, D",0.496,0.543,0.541
"gradient boosting, B",0.548,0.551,0.567
"gradient boosting, D",0.464,0.536,0.543
"logistic regression, B, trained on the stricter label",0.756,0.589,0.599


Top 50s of the model trained on the stricter label: 98% ROSE in late March (median slide -120%), 100% had 0 clicks in March, median 6 impressions per day


**The top of the queue is heavy with carry-over.** Over the whole pool, 13.4% of the declined pages did not fall another 20% after 31 March. In the top 50s that share is much higher: 43.0% of the W05 model's hits, 36.1% under D, 45.4% for the rule and 51.5% for the slide-only sort. These pages count as hits because they had already fallen in late March and April stayed below the March average, not because a new fall was predicted.

**On the stricter label nothing from W05 beats random.** The base rate is 0.501 in the pool (0.508 as a mean of folds). The W05 model gets P@50 0.504 (ROC AUC 0.549). The rule (0.428) and the slide-only sort (0.388) are even *below* random: pages that already slid tend to bounce back a bit rather than keep falling.

**A model trained for the stricter label finds something, but not refresh work.** It reaches P@50 0.756 (ROC AUC 0.599), but 98% of its top 50s are pages that *rose* in late March, all with zero clicks and a median of 6 impressions a day. It finds small spikes fading, which no refresh can prevent.

**What this means:** my queue is good at finding pages that already lost visibility in late March and did not recover in April. It is not evidence that the model sees a fall coming before it starts. For a review queue that is still useful (the reviewer looks at pages that are already down), but the claims have to say so.

### 3e. The attack checklist, filled in

In [20]:
# 3e. The attack checklist from the skill, filled in from the cells above
p50_B, p50_D = mean_main.loc[B_MAIN, "P@50"], mean_main.loc[D_MAIN, "P@50"]
checklist = pd.DataFrame([
    ("Timeline drawn: all features strictly before the label window", "PASS",
     f"features read up to {pd.Timestamp(MONTHS['2026-03']['last_date'].max()):%d %b}, the label starts {pd.Timestamp(MONTHS['2026-04']['first_date'].min()):%d %b}"),
    ("Filters do not select on the future", "PASS", f"rule 4 reads the label month but removes {RULE_4_REMOVED['Mar→Apr']} pages on its own"),
    ("No label-derived or sibling columns in the features", "PASS",
     f"W03 trap: 1.000 with the leak. Window-overlap trap here: P@50 {trap_table.iloc[2]['P@50']:.3f} with it, {trap_table.iloc[-1]['P@50']:.3f} without"),
    ("No product flags / existing-system scores as features", "PASS", "5 raw GSC features only. The W04 rule is a baseline, never an input"),
    ("Split grouped by the repeating entity (and/or time-based)", "PASS, now both",
     f"W05 was grouped only. Time-aware + grouped (D): P@50 {p50_D:.3f} vs {p50_B:.3f}"),
    ("Base rate printed next to every metric", "PASS", "random-order row in every table"),
    ("Top feature sanity-checked", "PASS",
     f"late_half_share alone: ROC AUC {alone.loc['late_half_share', 'ROC AUC alone, mean of held-out folds']:.3f}, without it P@50 {drop_table.loc['without late_half_share', 'P@50']:.3f}"),
    ("Metrics out-of-fold, never in-sample", "PASS", "every score comes from a model that never saw that client (B, D) or that month (C, D)"),
    ("Label shares a window with the features (carry-over)", "NOTE",
     f"{carry['hits in the top 50s: ' + MAIN + ', B (W05)']:.0%} of the W05 top-50 hits did not fall another 20% after 31 March"),
], columns=["check", "result", "evidence"]).set_index("check")
checklist

,result,evidence
check,,
Timeline drawn: all features strictly before the label window,PASS,"features read up to 31 Mar, the label starts 0..."
Filters do not select on the future,PASS,rule 4 reads the label month but removes 0 pag...
No label-derived or sibling columns in the features,PASS,W03 trap: 1.000 with the leak. Window-overlap ...
No product flags / existing-system scores as features,PASS,5 raw GSC features only. The W04 rule is a bas...
Split grouped by the repeating entity (and/or time-based),"PASS, now both",W05 was grouped only. Time-aware + grouped (D)...
Base rate printed next to every metric,PASS,random-order row in every table
Top feature sanity-checked,PASS,"late_half_share alone: ROC AUC 0.316, without ..."
"Metrics out-of-fold, never in-sample",PASS,every score comes from a model that never saw ...
Label shares a window with the features (carry-over),NOTE,43% of the W05 top-50 hits did not fall anothe...


Everything on the skill's list passes. The one note is the carry-over: no future data gets in, but the label partly restates March, so precision at the top overstates how much the model forecasts. Section 4 fixes the claims that leaned on it.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

I took the boldest sentences from my W05 notebook (and one from W04), checked each one against this week's numbers and rewrote it with the claim ladder from the `writing-honest-claims` skill: observed, measured, directional, decision-support. The cell below prints the numbers each rewrite rests on, so the text and the numbers can't drift apart.

In [21]:
# The numbers each rewrite in the table below rests on (all measured above, held-out clients)
def p50(table, method): return f"{table.loc[method, 'P@50']:.3f}"
evidence = pd.DataFrame([
    ("1", "Mar→Apr", "W05 split (B): my model vs rule", f"{p50(mean_main, B_MAIN)} vs {p50(mean_main, RULE)}", f"{p50_main.loc[B_MAIN, 'folds above the rule']} of 5 folds"),
    ("1", "Mar→Apr", "time-aware + grouped (D): my model vs rule", f"{p50(mean_main, D_MAIN)} vs {p50(mean_main, RULE)}", f"{p50_main.loc[D_MAIN, 'folds above the rule']} of 5 folds"),
    ("1", "Apr→May", "time-aware + grouped (D): my model vs rule", f"{p50(mean_next, D_MAIN)} vs {p50(mean_next, RULE)}", f"{p50_next.loc[D_MAIN, 'folds above the rule']} of 5 folds"),
    ("2", "Mar→Apr", "D: my model vs slide only", f"{p50(mean_main, D_MAIN)} vs {p50(mean_main, SLIDE_ONLY)}", f"{p50_main.loc[D_MAIN, 'folds above slide only']} of 5 folds"),
    ("2", "Apr→May", "D: my model vs slide only", f"{p50(mean_next, D_MAIN)} vs {p50(mean_next, SLIDE_ONLY)}", f"{p50_next.loc[D_MAIN, 'folds above slide only']} of 5 folds"),
    ("3", "Mar→Apr", "W05 top-50 hits that did not fall another 20% after 31 March", f"{carry['hits in the top 50s: ' + MAIN + ', B (W05)']:.1%}", ""),
    ("3", "Mar→Apr", "stricter label: my model (B) vs random", f"{strict_table.loc[B_MAIN, 'P@50']:.3f} vs {strict_table.loc[RANDOM, 'P@50']:.3f}",
     f"ROC AUC {strict_table.loc[B_MAIN, 'ROC AUC']:.3f}"),
    ("4", "Mar→Apr", "gradient boosting vs logistic regression, D: P@50 and lowest fold",
     f"{p50(mean_main, f'{CHECK}, D')} vs {p50(mean_main, D_MAIN)}",
     f"lowest fold {p50_main.loc[f'{CHECK}, D'].iloc[:N_FOLDS].min():.2f} vs {p50_main.loc[D_MAIN].iloc[:N_FOLDS].min():.2f}"),
    ("4", "Apr→May", "gradient boosting vs logistic regression, D", f"{p50(mean_next, f'{CHECK}, D')} vs {p50(mean_next, D_MAIN)}",
     f"lowest fold {p50_next.loc[f'{CHECK}, D'].iloc[:N_FOLDS].min():.2f} vs {p50_next.loc[D_MAIN].iloc[:N_FOLDS].min():.2f}"),
], columns=["claim", "test month", "comparison", "P@50 (or share)", "detail"])
evidence.set_index(["claim", "test month"])

comparison  \
claim test month                                                      
1     Mar→Apr                       W05 split (B): my model vs rule   
      Mar→Apr            time-aware + grouped (D): my model vs rule   
      Apr→May            time-aware + grouped (D): my model vs rule   
2     Mar→Apr                             D: my model vs slide only   
      Apr→May                             D: my model vs slide only   
3     Mar→Apr     W05 top-50 hits that did not fall another 20% ...   
      Mar→Apr                stricter label: my model (B) vs random   
4     Mar→Apr     gradient boosting vs logistic regression, D: P...   
      Apr→May           gradient boosting vs logistic regression, D   

                 P@50 (or share)                    detail  
claim test month                                            
1     Mar→Apr     0.884 vs 0.784              5 of 5 folds  
      Mar→Apr     0.776 vs 0.784              3 of 5 folds  
      Apr→May     0.988 vs 0.884              5 of 5 folds  
2     Mar→Apr     0.776 vs 0.800              1 of 5 folds  
      Apr→May     0.988 vs 0.980              3 of 5 folds  
3     Mar→Apr              43.0%                            
      Mar→Apr     0.504 vs 0.508             ROC AUC 0.549  
4     Mar→Apr     0.760 vs 0.776  lowest fold 0.48 vs 0.38  
      Apr→May     0.956 vs 0.988  lowest fold 0.88 vs 0.98

| # | My original sentence | What goes too far | Rewrite |
|---|---|---|---|
| 1 | W05: "**Logistic regression beats the rule on every fold.**" and "my logistic regression finds more pages that go on to lose 20%+ of their impressions (P@50 0.884 vs 0.784, higher on all 5 folds)" | Measured on held-out clients, but the model trained on April outcomes, which nobody has on 31 March. Trained only on the past it tied the rule in March→April (0.776 vs 0.784, above it on 3 of 5 folds) | "Measured on held-out clients and trained only on earlier months, the model's top 50 had about the same share of pages that went on to lose 20%+ as the W04 rule in March→April (0.776 vs 0.784) and a higher share in April→May (0.988 vs 0.884, all 5 folds). Directional: at least as good as the rule on this label, not reliably better." |
| 2 | W05: "So the model **adds about 8 points of P@50** at the top" (over the slide-only sort) | Only true with April outcomes in training. Trained on the past it adds nothing over the slide in either month pair | "Under a time-aware split the model did not add precision over sorting by the late-month slide alone (0.776 vs 0.800 in March→April, 0.988 vs 0.980 in April→May). The slide does most of the ranking." |
| 3 | W05: "The model is better at telling **which pages will drop**." | "Will drop" sounds like a forecast. 43% of its top-50 hits had already dropped in late March and did not fall another 20% in April. On a label that only counts a fall after 31 March it is at the base rate (0.504 vs 0.508) | "The model mostly ranks pages that had already lost impressions in late March. Measured on held-out clients, 43% of its top-50 hits only count because of that earlier drop. On a label that only counts a fall after 31 March it scores at the base rate. It is a decision-support list of pages worth a look first, not a forecast of new falls." |
| 4 | W04: "a page that is already losing impressions, or that gets fewer clicks than other pages at its position, is **the best early warning we found**" | "Early warning" suggests it sees trouble coming. Much of the effect is the late-month drop carrying into the label month | "In two month pairs, pages that were already losing impressions or had a low CTR for their position were observed to decline more often the next month. For the slide, much of this is the late-month drop carrying over into the monthly comparison. It is a cue for which pages to review first, not a forecast and not evidence that a refresh would help." |
| 5 | W05: "Gradient boosting was steadier across folds. **If that holds in W06, it is the one to switch to.**" | It did not hold. Under D its folds ran from 0.48 to 0.98 in March→April (W05: 0.84 to 0.94) and it scored below the logistic regression in both month pairs (0.760 vs 0.776, 0.956 vs 0.988) | "I keep the logistic regression: under a time-aware split gradient boosting was neither steadier nor better." |

The same rule applies to the paper findings in section 1: "older pages that were recently updated showed much higher impressions in this snapshot" is safe, "a refresh lifts health from 10.7 to 34.5" needs a before/after design first.

**What I take into W07 (the playbook):**
- Report the time-aware + grouped score (D) as the model's honest number, always next to the rule, the slide-only sort and a random queue.
- In both month pairs the slide-only sort is as good as the model at the top and needs no training. For the playbook I will put "slide only" and "model" side by side and keep the simpler one unless the model earns its place.
- The label rewards carry-over, so any queue built on it should say "already down", not "about to drop". For the label itself: try comparing each page with its own client's change (the W03 limitation) and keep `fell_after_decision` as a second check.
- Young pages and young, fast-growing clients are the main source of false alarms. A "full previous month: yes / no" column next to the queue would warn the reviewer.

In [22]:
# The run's receipt: small, no page or client ids, safe to commit (CSV files stay out of git by design)
def table_dict(t):
    return {str(m): {k: round(float(v), 4) for k, v in row.items()} for m, row in t.iterrows()}

receipt_out = {
    "notebook": "work/notebooks/w06_validation_audit.ipynb",
    "run_date": str(date.today()),
    "versions": VERSIONS,
    "seed": SEED,
    "model": {"name": MAIN, "features": FEATURES, "check_model": CHECK, "note": "W05 model unchanged (late_half_share = W05 late_march_share)"},
    "slices": {name: {"pages": int(len(d)), "clients": int(d["client_hash_id"].nunique()), "base_rate": round(float(d[LABEL].mean()), 4),
                      "pool_pages": int(d["in_pool"].sum()), "pool_base_rate": round(float(d.loc[d["in_pool"], LABEL].mean()), 4)}
               for name, d in FRAMES.items()},
    "designs": DESIGNS,
    "mar_apr_test": {"mean_over_folds": table_dict(mean_main), "p50_per_fold": {m: [round(float(v), 3) for v in row.iloc[:N_FOLDS]] for m, row in p50_main.iterrows()}},
    "apr_may_test": {"mean_over_folds": table_dict(mean_next), "p50_per_fold": {m: [round(float(v), 3) for v in row.iloc[:N_FOLDS]] for m, row in p50_next.iterrows()}},
    "leak_trap": table_dict(trap_table),
    "drop_one_feature": table_dict(drop_table),
    "carry_over_share": {k: round(float(v), 4) for k, v in carry.items()},
    "stricter_label": {"name": STRICT_LABEL, "pool_base_rate": round(float(pool[STRICT_LABEL].mean()), 4), "mean_over_folds": table_dict(strict_table)},
    "rule_4_removed_pages": RULE_4_REMOVED,
}
receipt_path = OUT_DIR / "w06_validation_audit_metrics.json"
OUT_DIR.mkdir(parents=True, exist_ok=True)
receipt_path.write_text(json.dumps(receipt_out, indent=2, ensure_ascii=False, default=str))
print(f"Wrote {receipt_path}")

Wrote ../outputs/w06_validation_audit_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Checked against "what done looks like":**

- [x] Names two paper findings and the methodology question for each, framed constructively: section 1 (the refresh effect and the growth model), with what the paper already does well and what would make each claim stronger
- [x] Re-runs my own model under a grouped and time-aware split with a before/after comparison: section 2 (A random rows, B grouped as in W05, C time-aware, D time-aware + grouped, all on the same test pages), plus a second month pair
- [x] Includes a leakage audit: section 3 (timeline, filters, window-overlap trap, train without each feature, label carry-over, the skill's checklist)
- [x] Includes error examples: section 2 (top 50 per fold, false alarms vs hits, the fold-4 client, the misses and three single pages with their weekly numbers)
- [x] All claims use public-safe language: section 4 rewrites my boldest W04 / W05 sentences. No client names, URLs or page / client ids in any output (clients appear by rank only)
- [x] Reproducible: seed 42, versions printed at the top, W04 / W05 numbers reproduced exactly, receipt written to `work/outputs/w06_validation_audit_metrics.json`